# HIGHFLY · PartCrafter GRATIS · Colab GPU T4 · Python 3.11 aislado (R0.4.2)

**Objetivo:** generar piezas 3D reales a partir de una imagen. El ZIP contiene `object.glb` y `part_XX.glb`; **todavía NO es armadura equipable** en ClaudeCraft y necesita revisión de arte + fitting/rig en Blender.

**ACTUALIZACIÓN IMPORTANTE 14/10/2026:** Colab ahora puede iniciarse con **Python 3.13**, incompatible con las ruedas de PyTorch 2.5.1/cu124 usadas por PartCrafter. Por eso la instalación anterior fallaba. Este cuaderno ahora instala **Python 3.11 aislado con uv**, sin tocar la versión de Python del cuaderno ni reiniciar Colab. La celda de inferencia ejecuta el intérprete 3.11.

**Costo:** cero créditos, cero API comerciales. Elegí solamente GPU T4 gratuita (mínimo 8 GB VRAM CUDA); su disponibilidad en Colab no está garantizada. No contratar servicios ni intentar sesiones automatizadas. El procesamiento puede tardar, agotar memoria o desconectarse.

**Licencias:** proyecto principal PartCrafter MIT. No se descarga ni utiliza el modelo de recorte de fondo RMBG-1.4 (tiene condiciones de uso diferentes); la imagen se centra en fondo blanco con Pillow. No se invoca Gemini ni otra IA comercial.

**Usar:** GPU T4 → primera celda → instalación corregida (puede tardar varios minutos) → subir referencia → generación → descarga ZIP → enviar a Guine.


In [ ]:
import os, sys, subprocess, platform
import torch
print("Python",sys.version.split()[0],"Torch",torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Colab NO asignó CUDA gratis. Revisá Entorno de ejecución > GPU. No contrates nada.")
props=torch.cuda.get_device_properties(0)
gb=props.total_memory / (1024**3)
print("GPU:",props.name, "| VRAM:",round(gb,2),"GiB")
if gb<7.5:
    raise RuntimeError("PartCrafter requiere al menos 8GB VRAM. No se ejecutará una inferencia imposible.")
assert os.environ.get("GEMINI_API_KEY") is None, "Para esta prueba no hacen falta claves de IA de pago"
print("GPU GRATIS compatible en principio; Colab puede cortar la sesión o agotar recursos.")


## 1. Instalar PartCrafter en Python 3.11 aislado (fix Colab 3.13 + APT)

Esta versión elimina el paso de `apt-get` que fallaba con código 100 aun después de instalar PyTorch. Como exportamos GLB sin renderizar video, no vamos a forzar bibliotecas gráficas innecesarias. Comprueba CUDA y los imports del generador. Si ya ejecutaste la celda anterior y llegó hasta `apt-get`, las dependencias ya están instaladas; podés volver a ejecutar esta celda, pero para ahorrar tiempo también podés correr únicamente el bloque de comprobación final.


In [ ]:
import os, sys, subprocess
from pathlib import Path

def run(cmd, cwd=None, timeout=900):
    print(">", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str,cmd)), cwd=cwd, check=True, timeout=timeout)

root = Path("/content/PartCrafter")
venv_path = Path("/content/highfly_partcrafter_py311")
pc_python = venv_path / "bin/python"

if not root.exists():
    run(["git", "clone", "--no-checkout", "https://github.com/wgsxm/PartCrafter.git", str(root)])
run(["git", "checkout", "3d773bf02fad51c7ab31a5615573fec93b287b30"], cwd=root)

# El cuaderno Colab queda con Python 3.13; las órdenes siguientes usan 3.11.
run([sys.executable, "-m", "pip", "install", "-q", "uv"], timeout=300)
run(["uv", "python", "install", "3.11"], timeout=600)
if not pc_python.exists():
    run(["uv", "venv", "--seed", "--python", "3.11", str(venv_path)], timeout=300)
run([str(pc_python), "-c", "import sys; assert sys.version_info[:2]==(3,11), sys.version; print('HIGHFLY PYTHON',sys.version.split()[0])"])

# Paquetes de GPU del proyecto oficial, ahora sí en el Python compatible.
run([str(pc_python), "-m", "pip", "install", "-q", 
     "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
     "--index-url", "https://download.pytorch.org/whl/cu124"], timeout=1600)

# PyTorch Geometric distribuye ruedas torch-cluster compiladas para esta combinación.
# No compilar desde fuente (evita largas compilaciones y errores ocultos).
run([str(pc_python), "-m", "pip", "install", "-q", "--no-cache-dir", "--only-binary=:all:",
     "torch-cluster", "-f", "https://data.pyg.org/whl/torch-2.5.1+cu124.html"], timeout=700)

# Mismas dependencias de la receta oficial; sin usar el pip del Python 3.13.
run([str(pc_python), "-m", "pip", "install", "-q", "-r", "settings/requirements.txt"],cwd=root, timeout=1800)

# PartCrafter exportará GLB sin --render; no bloquear por apt-get de Colab.
# La instalación del paquete libglu1-mesa puede fallar con exit 100 en Colab
# aunque Torch, torch-cluster y PartCrafter estén correctamente instalados.
# Verificamos el intérprete CUDA y los imports reales del generador.
run([str(pc_python), "-c",
     "import sys,torch,torch_cluster,trimesh; assert sys.version_info[:2]==(3,11); assert torch.cuda.is_available(); print('HIGHFLY CUDA OK:', torch.__version__, torch.cuda.get_device_name(0))"], timeout=120)
run([str(pc_python), "-c",
     "from src.pipelines.pipeline_partcrafter import PartCrafterPipeline; from src.utils.data_utils import get_colored_mesh_composition; from src.utils.render_utils import render_views_around_mesh; print('HIGHFLY PARTICLE IMPORTS OK')"], cwd=root, timeout=120)
print("HIGHFLY ✅ PartCrafter instalado en Python 3.11. Seguimos con la imagen. No hace falta reiniciar Colab.")


## 2. Evitar dependencia RMBG no comercial y preparar imagen

PartCrafter carga RMBG aun cuando no se pide remover fondo. Esta celda hace un cambio acotado y comprobado para omitir esa dependencia. La imagen se centra sobre **fondo blanco**, sin IA adicional. Es una prueba: ilustraciones con mucho fondo o una figura entera pueden producir piezas que no sean vestimentas.


In [ ]:
from pathlib import Path
from PIL import Image,ImageOps
from google.colab import files
src=(root/"scripts/inference_partcrafter.py")
txt=src.read_text(encoding="utf-8")
old_a='    snapshot_download(repo_id="briaai/RMBG-1.4", local_dir=rmbg_weights_dir)'
new_a='    if args.rmbg:\n        snapshot_download(repo_id="briaai/RMBG-1.4", local_dir=rmbg_weights_dir)'
old_b='    rmbg_net = BriaRMBG.from_pretrained(rmbg_weights_dir).to(device)\n    rmbg_net.eval()'
new_b='    rmbg_net = None\n    if args.rmbg:\n        rmbg_net = BriaRMBG.from_pretrained(rmbg_weights_dir).to(device)\n        rmbg_net.eval()'
if old_a in txt and old_b in txt:
    txt=txt.replace(old_a,new_a).replace(old_b,new_b)
    src.write_text(txt,encoding="utf-8")
elif new_a not in txt or new_b not in txt:
    raise RuntimeError("La fuente PartCrafter cambió. Se detiene antes de usar RMBG; revisar implementación.")
uploaded=files.upload()
if len(uploaded)!=1:raise ValueError("Subí UNA imagen PNG/JPG de armadura, no archivos privados ni múltiples imágenes.")
name,next_data=next(iter(uploaded.items()))
if not name.lower().endswith((".png",".jpg",".jpeg",".webp")):
    raise ValueError("Solo imagen PNG/JPEG/WEBP")
from io import BytesIO
image=Image.open(BytesIO(next_data)).convert("RGBA")
# Mantener silueta y evitar IA RMBG: transparencia verdadera si estaba en PNG.
background=Image.new("RGBA",image.size,(255,255,255,255))
background.alpha_composite(image)
image=background.convert("RGB")
image.thumbnail((768,768))
canvas=Image.new("RGB",(768,768),"white")
canvas.paste(image,((768-image.width)//2,(768-image.height)//2))
prepared=Path("/content/highfly_obsidian_reference.png")
canvas.save(prepared)
display(canvas.resize((384,384)))
print("Imagen lista (no hay bodypaint ni recolor):",prepared)


## 3. Generar geometrías independientes (si hay GPU T4 suficiente)

La generación usa **Python 3.11 aislado**; el cuaderno de Colab sigue en Python 3.13. Intentaremos 4 partes, 512 tokens y 25 pasos para ahorrar memoria. Esto **no** asegura calidad, separación semántica ni compatibilidad directa con el rig de ClaudeCraft. No consume créditos comerciales.


In [ ]:
from pathlib import Path
import subprocess, sys, os
assert pc_python.exists(), "Falta ejecutar la instalación de Python 3.11"
tag="highfly_obsidian_pilot"
outdir=Path("/content/highfly_partcrafter_output")
cmd=[str(pc_python),"scripts/inference_partcrafter.py",
 "--image_path",str(prepared),"--num_parts","4",
 "--num_tokens","512","--num_inference_steps","25",
 "--output_dir",str(outdir),"--tag",tag,"--seed","17"]
assert "--rmbg" not in cmd and "--part_suggest" not in cmd and "--style_transfer" not in cmd
subprocess.run(cmd,cwd=root,check=True,timeout=2400,env=dict(os.environ,GEMINI_API_KEY=""))
target=outdir/tag
assert (target/"object.glb").is_file(), "PartCrafter no produjo GLB, nada para importar."
pieces=sorted(target.glob("part_*.glb"))
assert len(pieces)==4 and all(x.stat().st_size>1000 for x in pieces)
print("GLB PARTES:",[(p.name,p.stat().st_size) for p in pieces])


## 4. Comprobar archivos y descargar ZIP para HIGHFLY

El ZIP contiene geometrías generadas y procedencia, sin credenciales. El siguiente paso es **auditar calidad, asignar piezas a slots y pesar en Blender**. No lo vamos a publicar como armadura lista.


In [ ]:
import json, zipfile,hashlib,datetime
from pathlib import Path
from google.colab import files
source_files=[target/"object.glb",target/"manifest.json",*sorted(target.glob("part_*.glb"))]
for p in source_files:
    assert p.is_file() and p.stat().st_size>30, f"Falta {p.name}"
meta={
 "schema":"highfly.partcrafter.raw-3d/v1",
 "provider":"PartCrafter MIT - image to multi-part 3D",
 "repository":"wgsxm/PartCrafter",
 "commit":"3d773bf02fad51c7ab31a5615573fec93b287b30",
 "gpu":torch.cuda.get_device_name(0),
 "input":"user-supplied image, white background; NO RMBG, NO paid VLM",
 "generatedParts":4,"riggedToClaudeCraft":False,"artApproved":False,
 "fileSha256":{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files}}
meta_file=target/"highfly-provenance.json"
meta_file.write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding="utf-8")
zip_path=Path("/content/HIGHFLY_OBSIDIAN_PARTCRAFTER_RAW3D.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in [*source_files,meta_file]:z.write(p,arcname=p.name)
print("Archivo real:",zip_path, "(",zip_path.stat().st_size,"bytes )")
files.download(str(zip_path))
